# ⚡ gdrive-zip-extractor — Selective ZIP Extractor for Google Colab

Peek inside a ZIP archive stored on Google Drive, pick exactly the files you want, and stream them out with live progress bars — without ever extracting the whole archive.

**How to use:**
1. Run **Cell 1** to mount your Google Drive.
2. Edit `ZIP_FILE` and `OUTPUT_DIR` at the top of **Cell 2**, then run it.
3. When prompted, type the numbers of the files you want (e.g. `1,3,7`).


## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import zipfile
import os
import time
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

print("🔗 Connecting to Google Drive...")
drive.mount("/content/drive")

print("✅ Google Drive mounted!\n")


## 2️⃣ Browse, Select & Extract

In [ ]:
import zipfile
import os
import time
from tqdm.auto import tqdm

# ============================================================
# CONFIGURATION
# ============================================================

ZIP_FILE = "/content/drive/MyDrive/your_archive.zip"   # 👈 point this at your ZIP file

# Extract to Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/extracted"          # 👈 where files will land

CHUNK_SIZE = 16 * 1024 * 1024  # 16 MB


# ============================================================
# SHOW ZIP CONTENTS
# ============================================================

with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:

    all_files = [
        f for f in zip_ref.infolist()
        if not f.is_dir()
    ]

    print("\n" + "=" * 70)
    print("📦 FILES INSIDE ZIP")
    print("=" * 70)

    for i, file in enumerate(all_files, 1):

        size_gb = file.file_size / (1024 ** 3)

        print(
            f"[{i:2}]  {file.filename}"
            f"   ({size_gb:.2f} GB)"
        )

    print("=" * 70)

    # --------------------------------------------------------
    # SELECT FILES
    # --------------------------------------------------------

    selection = input(
        "\n🎯 Enter file numbers to extract "
        "(example: 1,3,7): "
    )

    selected_numbers = [
        int(x.strip())
        for x in selection.split(",")
        if x.strip()
    ]

    selected_files = []

    for number in selected_numbers:

        if 1 <= number <= len(all_files):
            selected_files.append(
                all_files[number - 1]
            )
        else:
            print(
                f"⚠️ Ignoring invalid number: {number}"
            )


# ============================================================
# CONFIRM SELECTION
# ============================================================

print("\n" + "=" * 70)
print("🎯 SELECTED FILES")
print("=" * 70)

total_selected_size = 0

for file in selected_files:

    size_gb = file.file_size / (1024 ** 3)
    total_selected_size += file.file_size

    print(f"📄 {file.filename} — {size_gb:.2f} GB")

print("=" * 70)

print(
    f"📦 Files selected : {len(selected_files)}"
)

print(
    f"💾 Total size     : "
    f"{total_selected_size / (1024 ** 3):.2f} GB"
)


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# EXTRACT SELECTED FILES
# ============================================================

with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:

    overall_start = time.time()

    for index, info in enumerate(selected_files, 1):

        output_path = os.path.join(
            OUTPUT_DIR,
            info.filename
        )

        os.makedirs(
            os.path.dirname(output_path),
            exist_ok=True
        )

        print("\n" + "─" * 70)
        print(
            f"🚀 EXTRACTING {index}/{len(selected_files)}"
        )
        print(f"📄 {info.filename}")
        print(
            f"💾 {info.file_size / (1024 ** 3):.2f} GB"
        )
        print("─" * 70)

        start = time.time()

        with zip_ref.open(info, "r") as source:

            with open(output_path, "wb") as target:

                with tqdm(
                    total=info.file_size,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc="⚡ Progress",
                    colour="green",
                    dynamic_ncols=True
                ) as progress:

                    while True:

                        chunk = source.read(CHUNK_SIZE)

                        if not chunk:
                            break

                        target.write(chunk)
                        progress.update(len(chunk))

        elapsed = time.time() - start

        speed = (
            info.file_size /
            elapsed /
            (1024 ** 2)
        )

        print(
            f"✅ Complete | "
            f"⚡ {speed:.1f} MB/s | "
            f"⏱️ {elapsed / 60:.1f} min"
        )


# ============================================================
# DONE
# ============================================================

total_time = time.time() - overall_start

print("\n")
print("=" * 70)
print("🎉 SELECTED FILES EXTRACTED SUCCESSFULLY!")
print("=" * 70)

print(
    f"📦 Files extracted : {len(selected_files)}"
)

print(
    f"💾 Total data      : "
    f"{total_selected_size / (1024 ** 3):.2f} GB"
)

print(
    f"⏱️ Total time      : "
    f"{total_time / 60:.2f} minutes"
)

print(
    f"📂 Location        : {OUTPUT_DIR}"
)

print("=" * 70)
